### Microservicio

In [3]:
!pip install flask pyngrok

In [4]:
import os, pickle, threading, traceback
import numpy as np
import pandas as pd
from flask import Flask, request, jsonify
from pyngrok import ngrok, conf
from google.colab import userdata

# ---------- ngrok ----------
conf.get_default().auth_token = userdata.get('NGROK')
ngrok.kill()
public_url = ngrok.connect(5000).public_url
print(f" + ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:5000/\"")

# ---------- Load artifacts ONCE ----------
with open('/content/mdRgLnOver.pkl', 'rb') as f:
    modelo_dict = pickle.load(f)
with open('/content/modelOnehotencoder.pkl', 'rb') as f:
    oneHot = pickle.load(f)
with open('/content/labelEncoder.pkl', 'rb') as f:
    labelEnc = pickle.load(f)
with open('/content/columnas_modelo.pkl', 'rb') as f:
    columnas_modelo = pickle.load(f)

# Resolve the real estimator
modelo = modelo_dict['modelo'] if isinstance(modelo_dict, dict) and 'modelo' in modelo_dict else modelo_dict
columnas_necesarias = modelo_dict.get('columnas_entrenamiento') if isinstance(modelo_dict, dict) else None

# ---------- Flask ----------
app = Flask(__name__)
app.config["BASE_URL"] = public_url

REQUIRED_FIELDS = ['especialidad', 'sede', 'tipo_atencion', 'dia_semana', 'franja']

@app.route("/predict", methods=['POST'])
def index():
    try:
        data = request.get_json(silent=True)
        if not data:
            return jsonify({"error": "No se recibieron datos JSON"}), 400

        missing = [c for c in REQUIRED_FIELDS if c not in data]
        if missing:
            return jsonify({"error": f"Faltan campos: {missing}"}), 400

        datos_prueba = pd.DataFrame([{k: data[k] for k in REQUIRED_FIELDS}])

        # Ensure correct column order for the encoder
        datos_prueba = datos_prueba[list(oneHot.feature_names_in_)]

        # Encode
        x_encoded = oneHot.transform(datos_prueba)
        x_df = pd.DataFrame(x_encoded, columns=oneHot.get_feature_names_out())

        # Reorder for the model
        x_df = x_df[columnas_modelo]

        # Predict
        pred  = int(modelo.predict(x_df)[0])
        proba = modelo.predict_proba(x_df)[0]

        etiqueta = labelEnc.inverse_transform([pred])[0]
        prob_map = dict(zip(labelEnc.classes_, proba))

        resultado = {
            "prevision": etiqueta,
            "probabilidad": round(float(prob_map.get('yes', proba[pred])) * 100, 4),
            "probabilidades": {k: round(float(v) * 100, 4) for k, v in prob_map.items()},
        }
        return jsonify(resultado)

    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500


@app.route("/health", methods=['GET'])
def health_check():
    return jsonify({"status": "healthy", "model_loaded": modelo is not None}), 200


if __name__ == "__main__":
    threading.Thread(
        target=app.run,
        kwargs={"host": "0.0.0.0", "port": 5000, "debug": False, "use_reloader": False},
        daemon=True,
    ).start()
    print(f"API disponible en: {public_url}/predict")

 + ngrok tunnel "https://quintin-unresisted-jimply.ngrok-free.dev" -> "http://127.0.0.1:5000/"
API disponible en: https://quintin-unresisted-jimply.ngrok-free.dev/predict


In [5]:
from pyngrok import ngrok
ngrok.kill()